# K2 evaluation (EXPLORATORY)
CPU fake-model default. Step 1 runs all three arms; step 2 requires a Claude gate file bound to the downloaded step-1 manifest. Issuer wording is provisional pending L1 coordination. Values are exported for separately authorized Haiku grading; no grader call is made here.


In [ ]:
from pathlib import Path
import sys, json, hashlib, os
DRY_RUN = True
REPO = Path('/kaggle/input/taa-k2-repo/taa-lab')
# For local CPU dry-run only, set REPO to the checkout root.
PINNED_COMMIT = ''  # Full reviewed K2 commit; dataset metadata must match.
MODEL_PATH = Path('/kaggle/input/gemma-model/transformers/default/1')
MODEL_REVISION = ''  # Exact upstream snapshot revision, recorded by Chris.
OUTPUT = Path('/kaggle/working/k2-output')
if not DRY_RUN and (len(PINNED_COMMIT)!=40 or len(MODEL_REVISION)!=40):
    raise ValueError('Pin repo and model revisions before running')
if not DRY_RUN:
    assert (REPO/'COMMIT.txt').read_text().strip()==PINNED_COMMIT
sys.path.insert(0,str(REPO))
from exploratory.k2_kaggle import core
core.validate_assets()
OUTPUT.mkdir(parents=True,exist_ok=False)
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
metadata = dict(dry_run=DRY_RUN, repo_commit=PINNED_COMMIT, model_revision=MODEL_REVISION,
                asset_hashes=core.validate_assets())
def retain(row):
    with (OUTPUT/'events.jsonl').open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
STEP=1
REPEATS=1
GATE_FILE=None
ADAPTERS={'neutral':Path('/kaggle/input/k2-neutral/adapter'),'contrary':Path('/kaggle/input/k2-contrary/adapter')}
metadata.update(step=STEP,repeats=REPEATS,sampling=dict(do_sample=True,temperature=1.0,top_p=1.0),grader_max_tokens=1024)
gate=json.loads(Path(GATE_FILE).read_text()) if GATE_FILE else None


In [ ]:
if not DRY_RUN:
    import torch, transformers, peft
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
    metadata['gpu'] = torch.cuda.get_device_name(0)
    metadata['packages'] = {k:getattr(v,'__version__','unknown') for k,v in dict(torch=torch,transformers=transformers,peft=peft).items()}
    tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=False)
    model=AutoModelForCausalLM.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=False,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16),device_map={'':0})
    def prompt(messages):
        # Gemma 3 1B has no system role: fold system text into first user message.
        messages=[dict(x) for x in messages]
        if messages and messages[0]['role']=='system':
            system=messages.pop(0)['content']
            messages[0]['content']=system+'\n\n'+messages[0]['content']
        return tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)


In [ ]:
generate=core.fake_model
if not DRY_RUN:
    from peft import PeftModel
    # Attach fresh K2 adapters only, trained against this exact snapshot.
    model=PeftModel.from_pretrained(model,ADAPTERS['neutral'],adapter_name='neutral',is_trainable=False)
    model.load_adapter(ADAPTERS['contrary'],adapter_name='contrary',is_trainable=False)
    model.eval()
    metadata['adapters']={arm:{str(p.relative_to(path)):hashlib.sha256(p.read_bytes()).hexdigest()
        for p in path.rglob('*') if p.is_file()} for arm,path in ADAPTERS.items()}
    def generate(system,history,cap,seed,arm):
        set_seed(seed)
        messages=([dict(role='system',content=system)] if system else [])+history
        ids=prompt(messages)
        if len(ids)+cap>16384: raise ValueError('Evaluation context exceeds 16384; no truncation')
        inputs=torch.tensor([ids],device='cuda:0')
        def complete():
            with torch.no_grad():
                result=model.generate(inputs,max_new_tokens=cap,do_sample=True,temperature=1.0,top_p=1.0,
                    eos_token_id=[tokenizer.eos_token_id,tokenizer.convert_tokens_to_ids('<end_of_turn>')],
                    pad_token_id=tokenizer.pad_token_id)
            raw_ids=result[0,len(ids):].tolist()
            retain(dict(event='native_tokens',seed=seed,input_ids=ids,output_ids=raw_ids))
            return tokenizer.decode(raw_ids,skip_special_tokens=True)
        if arm=='base':
            with model.disable_adapter(): return complete()
        model.set_adapter(arm)
        return complete()
try:
    core.run(STEP,generate,retain,REPEATS,gate=gate,dry_run=DRY_RUN)
finally:
    if STEP==1:
        rows=[json.loads(s) for s in (OUTPUT/'events.jsonl').read_text().splitlines()] if (OUTPUT/'events.jsonl').exists() else []
        (OUTPUT/'sanity-summary.json').write_text(json.dumps(core.sanity_summary(rows,REPEATS),indent=2))
    core.manifest(OUTPUT,metadata)


## Values grading handoff
Export the 16 values responses per checkpoint and repeat under blinded identifiers. A separately approved Haiku run must use the existing `k1_values.RUBRIC`, strict JSON validation, and **max_tokens=1024** fixed before grading. Keep raw content blocks, stop reason, usage, invalid grades and refusals. Do not call `k1_values.score` directly: its historical cap is 256. No API token belongs in this notebook.
